In [1]:
import pandas as pd
import pyodbc
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
import xgboost as xgb
import numpy as np
from catboost import CatBoostRegressor as cbg
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import joblib

conn_string = r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=E:\DOWNLOAD 1\Aviator_tracker1.accdb;"

connection=pyodbc.connect(conn_string)
df=pd.read_sql("SELECT * FROM MultiplierData", connection)

df["Timestamp"]=pd.to_datetime(df["Timestamp"])

df['hour'] = df['Timestamp'].dt.hour
df['minute'] = df['Timestamp'].dt.minute
df['second'] = df['Timestamp'].dt.second
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df["WinRate"] = df["WonBets"] / df["TotalBets"]
df["BadRate"] = df["BadBets"] / df["TotalBets"]

df["rolling_win_mean"] = df["WonBets"].rolling(window=2).mean()
df["rolling_loss_std"] = df["LosersCount"].rolling(window=2).std()
df["rolling_win_max"] = df["WonBets"].rolling(window=2).max()
df["rolling_loss_max"] = df["LosersCount"].rolling(window=2).max()
df["rolling_win_median"] = df["WonBets"].rolling(window=2).median()
df["rolling_loss_median"] = df["LosersCount"].rolling(window=2).median()

df.head()


C:\Users\user\AppData\Local\Temp\ipykernel_10224\789753488.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql("SELECT * FROM MultiplierData", connection)
C:\Users\user\AppData\Local\Temp\ipykernel_10224\789753488.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Timestamp"]=pd.to_datetime(df["Timestamp"])


,ID,Multiplier,CashoutVALUE,WonBets,BadBets,TotalBets,LosersCount,Timestamp,hour,minute,...,hour_sin,hour_cos,WinRate,BadRate,rolling_win_mean,rolling_loss_std,rolling_win_max,rolling_loss_max,rolling_win_median,rolling_loss_median
0,335,1.47,61401.93,367,8048,8415,4840,2025-07-05 19:24:54,19,24,...,-0.965926,0.258819,0.043613,0.956387,NaN,NaN,NaN,NaN,NaN,NaN
1,336,1.13,0,130,5194,5324,3358,2025-07-05 19:25:06,19,25,...,-0.965926,0.258819,0.024418,0.975582,248.5,1047.932250,367.0,4840.0,248.5,4099.0
2,337,1.60,129920.07,446,6135,6581,3897,2025-07-05 19:25:35,19,25,...,-0.965926,0.258819,0.067771,0.932229,288.0,381.130555,446.0,3897.0,288.0,3627.5
3,338,1.19,80235.3,191,7768,7959,4689,2025-07-05 19:25:48,19,25,...,-0.965926,0.258819,0.023998,0.976002,318.5,560.028571,446.0,4689.0,318.5,4293.0
4,339,1.88,178332.49,663,6325,6988,4019,2025-07-05 19:26:06,19,26,...,-0.965926,0.258819,0.094877,0.905123,427.0,473.761543,663.0,4689.0,427.0,4354.0


In [2]:
def lagged_structure(data, n_lags=50, target_field='Multiplier'):
    data=data.drop(columns=["Timestamp", "ID"])
    df=data.copy()
    df['CashoutVALUE'] = pd.to_numeric(df['CashoutVALUE'], errors='coerce')
    lagged_list=[]
    for lag in range(1, n_lags+1):
        shifted_columns=df.shift(lag)
        shifted_columns.columns=[f"{col}_lag{lag}" for col in shifted_columns.columns]
        lagged_list.append(shifted_columns)

    lagged_df=pd.concat(lagged_list+[df[[target_field]]], axis=1)
    lagged_df.dropna(inplace=True)

    X=lagged_df.drop(columns=[target_field])
    y=lagged_df[target_field]

    X=df.drop([target_field], axis=1)
    y=df[target_field]
    return X, y
    
X, y=lagged_structure(df, n_lags=50, target_field='Multiplier')
#print(X.head())

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, shuffle=False)

#XGB = xgb.XGBRegressor(objective='reg:squarederror')
catboost_model=cbg(random_state=42, verbose=0, depth=5, iterations=120, learning_rate=0.1)
xgboost_model=xgb.XGBRegressor(learning_rate=0.2, max_depth=3, n_estimators=50, objective='reg:squarederror')

meta_model=Ridge(alpha=1.0)

stacked = StackingRegressor(
    estimators=[
        ('xgb', xgboost_model),
        ('cat', catboost_model),
    ],
    final_estimator=meta_model,
    cv=5
)

stacked.fit(X_train, y_train)
y_pred=stacked.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"Stacked RMSE: {rmse:.4f}")

'''plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='royalblue', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)  # Diagonal line
plt.xlabel('Actual Multiplier')
plt.ylabel('Predicted Multiplier')
plt.title('Predicted vs. Actual Multiplier')
plt.grid(True)
plt.tight_layout()
plt.show()'''


Stacked RMSE: 13.8002


e:\ANACONDA\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


"plt.figure(figsize=(10, 6))\nplt.scatter(y_test, y_pred, alpha=0.5, color='royalblue', edgecolors='k')\nplt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)  # Diagonal line\nplt.xlabel('Actual Multiplier')\nplt.ylabel('Predicted Multiplier')\nplt.title('Predicted vs. Actual Multiplier')\nplt.grid(True)\nplt.tight_layout()\nplt.show()"

In [5]:
joblib.dump(stacked, "Avi_model13.8_v2.pkl")

['Avi_model13.8_v2.pkl']

In [ ]:
optimized_xgb_grid = {
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [3, 5],
    'subsample': [0.8],
    'reg_alpha': [0.1, 1],
    'colsample_bytree': [0.8, 1.0],
    "n_estimators":[100, 200, 500],
    "gamma":[0, 0.1, 0.2],
    "subsample":[0.6, 0.8, 1.0],
    "reg_alpha":[0, 0.1, 1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 5, 10]
}

optimized_cat_grid = {
    'learning_rate': [0.03, 0.05],
    'depth': [6, 8],
    'l2_leaf_reg': [3, 5],
    'border_count': [128],
    'random_strength': [0.5, 1],
}

grid_search_xgb = GridSearchCV(
    estimator=xgboost_model,
    param_grid=optimized_xgb_grid,
    scoring='neg_root_mean_squared_error',  # or 'r2', 'neg_mean_absolute_error', etc.
    cv=5,
    verbose=3,
    n_jobs=-1  # use all cores
)

grid_search_cat=GridSearchCV(
    estimator=cbg,
    param_grid=optimized_cat_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search_xgb.fit(X_train, y_train)
print(f"Params for XGB:{grid_search_xgb.best_params_}")
print(f"Best score (negative RMSE) XGB:{grid_search_xgb.best_score_}")

'''grid_search_cat.fit(X_train, y_train)
print(f"Params for CGB:{grid_search_cat.best_params_}")
print(f"Best score (negative RMSE) CGB:{grid_search_cat.best_score_}")'''
#score=cross_val_score(XGB, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
#print(score)

Fitting 5 folds for each of 4374 candidates, totalling 21870 fits


In [4]:
import sys
print(sys.executable)

e:\ANACONDA\anaconda3\python.exe
